# 🏛️ Hands-on: the Claude API, by actually using it

**Colab edition** of the [10 progressive exercises](https://github.com/) from the *Claude Certified Architect*
prep track. Each exercise has a **goal**, runnable **code**, what to **observe**, and the **learning**.
Run them **in order** — each builds on the last.

> 💰 **Cost:** everything here uses Haiku or short calls — the whole notebook costs a **few US cents**.
> 🔑 **Key:** you need an Anthropic API key ([how to get one](https://console.anthropic.com) — see the track's
> *Getting your API key* guide for billing setup and safety rules). The next cell asks for it with a **hidden
> prompt** — it is stored only in this runtime's memory, never in the notebook file.


In [ ]:
# Setup — run me first
%pip install -q anthropic

import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key (input is hidden): ")

import anthropic
client = anthropic.Anthropic()
print("Client ready ✓")

## Exercise 1 — Your first call (prove the doorway works)
**Goal:** send one message, get one reply.

**Observe:** a one-sentence answer prints. **Learning:** this is the whole API — one call, `content[0].text`
is the reply. A 401 error means your key isn't set correctly (re-run the setup cell).

In [ ]:
msg = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "In one sentence, what is an API?"}],
)
print(msg.content[0].text)

## Exercise 2 — Inspect the whole response object
**Goal:** see what *else* comes back besides the text.

**Observe:** `stop_reason` = `end_turn`; `usage` shows token counts (your bill). **Learning:** the reply is an
*object*, not just a string. You'll check `stop_reason` and `usage` constantly.

In [ ]:
msg = client.messages.create(
    model="claude-haiku-4-5", max_tokens=100,
    messages=[{"role": "user", "content": "Name three primary colors."}],
)
print("text:       ", msg.content[0].text)
print("stop_reason:", msg.stop_reason)          # why it stopped
print("usage:      ", msg.usage)                # input/output tokens (your bill)
print("model:      ", msg.model)

## Exercise 3 — Feel `max_tokens` cut a reply off
**Goal:** trigger a truncated response on purpose.

**Observe:** at 20 the text is cut mid-thought and `stop_reason == "max_tokens"`; at 500 it finishes with
`end_turn`. **Learning:** `max_tokens` caps the *reply*. Truncated output? Check `stop_reason`, raise the cap —
a real debugging reflex.

In [ ]:
for cap in (20, 500):
    msg = client.messages.create(
        model="claude-haiku-4-5", max_tokens=cap,
        messages=[{"role": "user", "content": "Explain how rainbows form."}],
    )
    print(f"\n--- max_tokens={cap} | stop_reason={msg.stop_reason} ---")
    print(msg.content[0].text)

## Exercise 4 — Temperature: repeatable vs varied
**Goal:** see determinism appear at `temperature=0`.

**Observe:** high temp → variety; temp 0 → (near-)identical each run. **Learning:** set `temperature=0`
whenever you need consistency (extraction, classification, evals). Leaving it high is why "the same prompt
gives different answers".

In [ ]:
def ask(temp):
    m = client.messages.create(
        model="claude-haiku-4-5", max_tokens=30, temperature=temp,
        messages=[{"role": "user", "content": "Invent a name for a coffee shop."}],
    )
    return m.content[0].text.strip()

print("temp=1.0:", [ask(1.0) for _ in range(3)])   # three different names
print("temp=0.0:", [ask(0.0) for _ in range(3)])   # nearly identical

## Exercise 5 — The `system` prompt changes behavior
**Goal:** the same question answered in two personas.

**Observe:** wildly different tone/level, same question. **Learning:** `system` is a *separate parameter*
(not a message) that sets role + rules — your main lever for controlling behavior.

In [ ]:
def ask(system):
    m = client.messages.create(
        model="claude-haiku-4-5", max_tokens=120, system=system,
        messages=[{"role": "user", "content": "What is a black hole?"}],
    )
    return m.content[0].text

print(ask("You explain things to a curious 5-year-old, with a simple analogy."))
print("\n---\n")
print(ask("You are an astrophysicist writing for a graduate seminar."))

## Exercise 6 — Multi-turn: prove statelessness yourself
**Goal:** show Claude "forgets" unless you resend history.

**Observe:** the first path can't recall the name; the second can. **Learning:** the API is **stateless** —
*you* own the conversation and resend it every turn. This is *why* long chats fill the context window.

In [ ]:
# WRONG way — no history carried:
client.messages.create(model="claude-haiku-4-5", max_tokens=50,
    messages=[{"role": "user", "content": "My cat is named Pip."}])
r = client.messages.create(model="claude-haiku-4-5", max_tokens=50,
    messages=[{"role": "user", "content": "What is my cat's name?"}])
print("no-history →", r.content[0].text)      # it does NOT know

# RIGHT way — carry the whole conversation:
history = [
    {"role": "user", "content": "My cat is named Pip."},
    {"role": "assistant", "content": "Nice to meet Pip!"},
    {"role": "user", "content": "What is my cat's name?"},
]
r = client.messages.create(model="claude-haiku-4-5", max_tokens=50, messages=history)
print("with-history →", r.content[0].text)     # → "Pip"

## Exercise 7 — Build a tiny chatbot
**Goal:** turn statelessness into a working chat loop (3 turns here — extend it if you like).

**Observe:** it remembers across turns because *you* keep appending. **Learning:** a chatbot is just:
append user turn → call → append assistant turn → repeat. That append **is** the entire "memory" —
exactly what claude.ai does for you invisibly.

In [ ]:
history = []
for turn in range(3):                       # 3 turns; change to while True for endless chat
    user = input(f"you ({turn+1}/3): ")
    history.append({"role": "user", "content": user})
    msg = client.messages.create(
        model="claude-haiku-4-5", max_tokens=300,
        system="You are a concise, friendly assistant.",
        messages=history,
    )
    reply = msg.content[0].text
    print("claude:", reply, "\n")
    history.append({"role": "assistant", "content": reply})   # ← append the reply!

print(f"history now holds {len(history)} messages — this is what gets resent every turn.")

## Exercise 8 — Stream the reply (typing effect)
**Goal:** see tokens arrive live.

**Observe:** text appears word-by-word instead of all at once. **Learning:** use `stream()` for chat UIs;
plain `create()` for backend/batch. Same request, different delivery.

In [ ]:
with client.messages.stream(
    model="claude-haiku-4-5", max_tokens=200,
    messages=[{"role": "user", "content": "Write a haiku about the ocean."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

## Exercise 9 — Handle errors like a grown-up
**Goal:** catch and read a real API error (a deliberately wrong model name).

**Observe:** a clean, catchable exception instead of a crash. **Learning:** the SDK raises typed errors
(`APIStatusError`, `RateLimitError`, `AuthenticationError`). 401 = key problem, 429 = rate limit,
400 = bad request, 404 = wrong model name.

In [ ]:
try:
    client.messages.create(
        model="claude-does-not-exist", max_tokens=10,
        messages=[{"role": "user", "content": "hi"}],
    )
except anthropic.APIStatusError as e:
    print("status:", e.status_code)   # 404 / 400
    print("body:  ", e.message)

## Exercise 10 — Mini use-case: a support-ticket classifier
**Goal:** combine everything into something real.

**Learning:** you just built a real classifier — Haiku (cheap, high-volume), `temperature=0` (repeatable),
`system` for the rules, small `max_tokens`. Every choice is an architect decision you can now *defend*.

In [ ]:
TICKETS = [
    "I was charged twice this month!",
    "The app crashes when I open settings.",
    "Please cancel my subscription.",
]

def classify(ticket):
    m = client.messages.create(
        model="claude-haiku-4-5", max_tokens=20, temperature=0,   # temp 0 = consistent
        system="Classify the support ticket into exactly one of: billing, technical, cancellation. "
               "Reply with ONLY that one word, lowercase.",
        messages=[{"role": "user", "content": ticket}],
    )
    return m.content[0].text.strip()

for t in TICKETS:
    print(f"{classify(t):12}  <- {t}")

---
## ✅ Checkpoint — you've "got it" when you can, without looking:

- [ ] Explain the difference between the **API**, the **SDK**, and **Claude Code**.
- [ ] Write a `messages.create` call from memory with model, max_tokens, system, messages.
- [ ] Say what `content[0].text`, `stop_reason`, and `usage` are.
- [ ] Explain why the chatbot needs you to **append** the assistant reply.
- [ ] Say when you'd set `temperature=0` and why.
- [ ] Predict what `stop_reason` you'll get if `max_tokens` is too small.

Miss any? Re-run that exercise. **Running it beats re-reading it.**

**Next on the track:** the *Prompt Engineering* domain — where Exercise 10's classifier grows a JSON schema,
few-shot examples, and an eval. Head back to the learning site → `illustrated/4-prompting/prompt-anatomy.html`.
